# 02 - Toy Embeddings with PyTorch: Visual Feedforward Walkthrough

This notebook trains a tiny missing-word model and then displays the complete numerical flow using visible grid cells.

```text
Word IDs -> embedding lookup -> mean pooling -> matrix multiplication
-> bias addition -> logits -> softmax probabilities
```

**No torchview or Graphviz installation is required.** The diagrams use `matplotlib` only.


## 1. Imports and reproducibility

In [ ]:
import re
import torch
from torch import nn
from torch.nn import functional as F
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, Circle, FancyArrowPatch

torch.manual_seed(42)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())


## 2. Create the toy corpus

In [ ]:
sentences = [
    "customers buy products",
    "customers purchase products",
    "people buy goods",
    "people purchase goods",
    "shoppers buy products",
    "shoppers purchase goods",
    "companies sell products",
    "stores sell goods",
    "customers visit stores",
    "shoppers visit stores",
    "dogs chase cats",
    "puppies chase cats",
    "dogs like food",
    "puppies like food",
] * 100

def tokenize(text):
    return re.findall(r"[a-z]+", text.lower())

tokenized = [tokenize(sentence) for sentence in sentences]
vocabulary = sorted({word for sentence in tokenized for word in sentence})
word_to_id = {word: index for index, word in enumerate(vocabulary)}
id_to_word = {index: word for word, index in word_to_id.items()}
vocabulary_size = len(vocabulary)

print("Vocabulary size:", vocabulary_size)
print(vocabulary)


## 3. Create missing-word training examples

For `customers buy products`:

```text
inputs = customers, products
label  = buy
```


In [ ]:
contexts = []
targets = []

for left_word, target_word, right_word in tokenized:
    contexts.append([word_to_id[left_word], word_to_id[right_word]])
    targets.append(word_to_id[target_word])

context_tensor = torch.tensor(contexts, dtype=torch.long)
target_tensor = torch.tensor(targets, dtype=torch.long)

print("Context tensor:", context_tensor.shape)
print("Target tensor:", target_tensor.shape)
print("Example IDs:", context_tensor[0])
print("Example words:", [id_to_word[i.item()] for i in context_tensor[0]])
print("Target:", id_to_word[target_tensor[0].item()])


## 4. Define the feedforward model

In [ ]:
class MissingWordModel(nn.Module):
    def __init__(self, vocabulary_size, embedding_dimension):
        super().__init__()
        self.embedding = nn.Embedding(vocabulary_size, embedding_dimension)
        self.output_layer = nn.Linear(embedding_dimension, vocabulary_size)

    def forward(self, context_word_ids):
        context_embeddings = self.embedding(context_word_ids)
        pooled_context = context_embeddings.mean(dim=1)
        logits = self.output_layer(pooled_context)
        return logits

embedding_dimension = 5
model = MissingWordModel(vocabulary_size, embedding_dimension)
print(model)

for name, parameter in model.named_parameters():
    print(name, tuple(parameter.shape), "trainable values:", parameter.numel())


## 5. Train with automatic backpropagation

In [ ]:
loss_function = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.05)
loss_history = []

for epoch in range(300):
    optimizer.zero_grad()
    logits = model(context_tensor)
    loss = loss_function(logits, target_tensor)
    loss.backward()
    optimizer.step()
    loss_history.append(loss.item())

    if epoch % 50 == 0:
        print(f"Epoch {epoch:3d} | Loss {loss.item():.4f}")

print("Final loss:", round(loss_history[-1], 4))


## 6. Diagram helper

Every rectangle below represents one scalar value. A row of five rectangles is a five-value vector. A five-by-seventeen rectangle grid is a matrix with five rows and seventeen columns.


In [ ]:
def draw_grid(ax, values, x, y, cell_w=0.65, cell_h=0.55,
              title=None, color="#D9EAF7", decimals=2, fontsize=7):
    tensor = torch.as_tensor(values).detach().cpu()
    if tensor.ndim == 0:
        tensor = tensor.reshape(1, 1)
    elif tensor.ndim == 1:
        tensor = tensor.unsqueeze(0)

    rows, columns = tensor.shape
    for row in range(rows):
        for column in range(columns):
            left = x + column * cell_w
            bottom = y - row * cell_h
            ax.add_patch(Rectangle(
                (left, bottom), cell_w, cell_h,
                facecolor=color, edgecolor="#17365D", linewidth=1
            ))
            value = tensor[row, column].item()
            label = str(int(value)) if not tensor.dtype.is_floating_point else f"{value:.{decimals}f}"
            ax.text(left + cell_w/2, bottom + cell_h/2, label,
                    ha="center", va="center", fontsize=fontsize)

    if title:
        ax.text(x + columns*cell_w/2, y + cell_h*0.65, title,
                ha="center", va="bottom", fontsize=10,
                fontweight="bold", color="#17365D")


def arrow(ax, start, end, label=""):
    ax.add_patch(FancyArrowPatch(start, end, arrowstyle="-|>",
                                 mutation_scale=14, linewidth=1.5,
                                 color="#2F75B5"))
    if label:
        ax.text((start[0]+end[0])/2, (start[1]+end[1])/2 + 0.18,
                label, ha="center", fontsize=8, color="#2F75B5")


## 7. Feedforward neural-network diagram

The five middle nodes are the five values in the pooled context vector. Each context value connects to all seventeen output logits because the output layer is fully connected.


In [ ]:
fig, ax = plt.subplots(figsize=(15, 8))
ax.set_xlim(0, 15); ax.set_ylim(0, 10); ax.axis("off")

input_positions = [(1.2, 6.4), (1.2, 3.6)]
context_positions = [(7.0, 8-i*1.35) for i in range(5)]
logit_positions = [(10.5, 8.8-i*0.45) for i in range(vocabulary_size)]
softmax_positions = [(13.5, y) for _, y in logit_positions]

for (x,y), label in zip(input_positions, ["customers ID", "products ID"]):
    ax.add_patch(Circle((x,y), .25, facecolor="#FFF2CC", edgecolor="#17365D"))
    ax.text(x, y-.5, label, ha="center", fontsize=8)

# embedding blocks as two rows of five squares
for row, word in enumerate(["customers embedding", "products embedding"]):
    yy = 6.85-row*2.8
    for col in range(5):
        ax.add_patch(Rectangle((3.1+col*.52, yy), .5, .5,
                               facecolor="#D9EAF7", edgecolor="#17365D"))
    ax.text(4.35, yy+.72, word, ha="center", fontsize=9)

for i,(x,y) in enumerate(context_positions):
    ax.add_patch(Circle((x,y), .22, facecolor="#E2F0D9", edgecolor="#17365D"))
    ax.text(x, y, f"c{i+1}", ha="center", va="center", fontsize=8)

for x,y in logit_positions:
    ax.add_patch(Circle((x,y), .11, facecolor="#FFF2CC", edgecolor="#17365D"))
for x,y in softmax_positions:
    ax.add_patch(Circle((x,y), .11, facecolor="#E2F0D9", edgecolor="#17365D"))

for src in input_positions:
    arrow(ax, (src[0]+.25, src[1]), (3.0, src[1]))
for y in [7.1, 4.3]:
    for target in context_positions:
        ax.plot([5.75, target[0]-.22], [y, target[1]], color="#A8B8C8", linewidth=.45)
for src in context_positions:
    for target in logit_positions:
        ax.plot([src[0]+.22, target[0]-.11], [src[1], target[1]], color="#B7C9D6", linewidth=.25)
for src,target in zip(logit_positions, softmax_positions):
    ax.plot([src[0]+.11, target[0]-.11], [src[1], target[1]], color="#7FA6C9", linewidth=.6)

ax.text(1.2, 9.5, "2 input IDs", ha="center", fontweight="bold")
ax.text(4.35, 9.5, "Embedding lookup
2 rows x 5 cells", ha="center", fontweight="bold")
ax.text(7, 9.5, "Mean pooling
5 cells", ha="center", fontweight="bold")
ax.text(10.5, 9.5, "Linear layer
17 logits", ha="center", fontweight="bold")
ax.text(13.5, 9.5, "Softmax
17 probabilities", ha="center", fontweight="bold")
ax.text(5.7, 1.0, "Average corresponding cells in the two embedding rows", ha="center", color="#2F75B5")
ax.text(9.0, .55, "Every pooled value contributes to every candidate-word logit", ha="center", color="#2F75B5")
plt.show()


## 8. Actual embedding lookup and averaging

The next diagram uses trained values from the current model. Each of the two words selects one row from the embedding matrix. The two rows are averaged column by column.


In [ ]:
example_words = ["customers", "products"]
example_ids = torch.tensor([word_to_id[w] for w in example_words])

with torch.no_grad():
    selected_embeddings = model.embedding(example_ids)
    pooled_context = selected_embeddings.mean(dim=0)

fig, ax = plt.subplots(figsize=(15,5))
ax.set_xlim(0,15); ax.set_ylim(0,5); ax.axis("off")

draw_grid(ax, example_ids, .4, 2.7, .7, title="2 word IDs", color="#FFF2CC", decimals=0)
draw_grid(ax, selected_embeddings, 3.0, 3.1, .82, .75,
          title="2 selected embedding rows, 5 values each", color="#D9EAF7")
draw_grid(ax, pooled_context, 10.0, 2.7, .82,
          title="5 averaged values", color="#E2F0D9")

ax.text(2.75, 3.48, "customers", ha="right", fontsize=9)
ax.text(2.75, 2.73, "products", ha="right", fontsize=9)
arrow(ax, (1.95,2.98), (2.75,2.98), "lookup")
arrow(ax, (7.3,2.98), (9.75,2.98), "average columns")
ax.text(7.5,.7, "For each column: (top embedding value + bottom embedding value) / 2",
        ha="center", fontsize=10, color="#17365D")
plt.show()

print("Selected embeddings:
", selected_embeddings)
print("
Pooled context:
", pooled_context)


## 9. Linear layer as matrix multiplication

PyTorch stores the output weights as seventeen rows by five columns. One row belongs to each candidate target word.

For matrix multiplication, PyTorch uses the transpose:

```text
[ five context cells ]
        multiplied by
[ five rows x seventeen columns ]
        produces
[ seventeen product cells ]
```

Then seventeen bias cells are added to produce seventeen logits.


In [ ]:
with torch.no_grad():
    weight = model.output_layer.weight.detach()       # 17 x 5
    weight_transpose = weight.T                       # 5 x 17
    bias = model.output_layer.bias.detach()           # 17
    product = pooled_context @ weight_transpose       # 17
    manual_logits = product + bias                    # 17
    model_logits = model(example_ids.unsqueeze(0))[0]

print("Manual calculation equals nn.Linear:", torch.allclose(manual_logits, model_logits))

fig, ax = plt.subplots(figsize=(20,11))
ax.set_xlim(0,20); ax.set_ylim(0,12); ax.axis("off")

draw_grid(ax, pooled_context, .3, 8.2, .65, title="Context row: 5 cells", color="#E2F0D9")
ax.text(4.0,8.45,"x",fontsize=22,fontweight="bold")
draw_grid(ax, weight_transpose, 4.6, 10.0, .52, .48,
          title="Weight transpose: 5 rows x 17 columns", color="#D9EAF7", fontsize=5)
ax.text(13.8,8.45,"=",fontsize=22,fontweight="bold")
draw_grid(ax, product, 14.4, 8.2, .30, title="17 dot products", color="#FFF2CC", fontsize=5)

draw_grid(ax, bias, 6.0, 5.0, .48, title="17 bias cells", color="#FCE4D6", fontsize=5)
ax.text(14.6,5.25,"+",fontsize=22,fontweight="bold")
draw_grid(ax, manual_logits, 6.0, 2.5, .48, title="17 final logits", color="#E2F0D9", fontsize=5)

ax.text(10, .65,
        "Each output logit = dot product of the five context values with one candidate-word weight row + that word's bias",
        ha="center", fontsize=10, color="#17365D")
plt.show()


## 10. Zoom into one output neuron: the word `buy`

The model calculates the `buy` logit using five pairwise multiplications, a sum, and one bias addition.


In [ ]:
candidate_word = "buy"
candidate_id = word_to_id[candidate_word]

with torch.no_grad():
    candidate_weights = weight[candidate_id]
    five_products = pooled_context * candidate_weights
    dot_product = five_products.sum()
    candidate_bias = bias[candidate_id]
    candidate_logit = dot_product + candidate_bias

fig, ax = plt.subplots(figsize=(17,6))
ax.set_xlim(0,17); ax.set_ylim(0,6); ax.axis("off")

draw_grid(ax, pooled_context, .3, 3.5, .7, title="5 context cells", color="#E2F0D9")
ax.text(4.05,3.75,"element-wise x",ha="center",fontweight="bold")
draw_grid(ax, candidate_weights, 5.0, 3.5, .7, title="5 'buy' weight cells", color="#D9EAF7")
ax.text(8.75,3.75,"=",fontsize=20,fontweight="bold")
draw_grid(ax, five_products, 9.2, 3.5, .7, title="5 multiplication results", color="#FFF2CC")
ax.text(13.0,3.75,"sum + bias",ha="center",fontweight="bold")
draw_grid(ax, candidate_logit, 14.5, 3.5, .9, title="buy logit", color="#F4B183")

ax.text(8.5,1.3,
        f"sum(products) = {dot_product.item():.4f}    bias = {candidate_bias.item():.4f}    logit = {candidate_logit.item():.4f}",
        ha="center", fontsize=11, color="#17365D")
plt.show()


## 11. Softmax as a visible transformation

The seventeen logits can be positive or negative. Softmax converts all seventeen values into non-negative probabilities whose total equals one.


In [ ]:
with torch.no_grad():
    probabilities = torch.softmax(manual_logits, dim=0)

fig, ax = plt.subplots(figsize=(16,6))
ax.set_xlim(0,16); ax.set_ylim(0,6); ax.axis("off")

draw_grid(ax, manual_logits, .3, 4.1, .72, title="17 logits", color="#FFF2CC", fontsize=6)
draw_grid(ax, probabilities, .3, 1.9, .72, title="17 softmax probabilities", color="#E2F0D9", decimals=3, fontsize=6)
arrow(ax, (13.0,4.35), (14.2,4.35), "softmax")

for index, word in enumerate(vocabulary):
    ax.text(.3 + index*.72 + .36, .95, word, rotation=60,
            ha="right", va="top", fontsize=6)

ax.text(14.3,2.2,f"Sum = {probabilities.sum().item():.4f}",fontsize=10)
plt.show()

print("Top predictions:")
top_p, top_i = torch.topk(probabilities, 5)
for p, i in zip(top_p, top_i):
    print(f"{id_to_word[i.item()]:10s} logit={manual_logits[i].item():8.4f} probability={p.item():.4f}")


## 12. Entire numerical flow in one view

In [ ]:
fig, ax = plt.subplots(figsize=(21,7))
ax.set_xlim(0,21); ax.set_ylim(0,7); ax.axis("off")

draw_grid(ax, example_ids, .2, 3.8, .6, title="2 IDs", color="#FFF2CC", decimals=0)
draw_grid(ax, selected_embeddings, 2.2, 4.15, .55, .6,
          title="2 x 5 embeddings", color="#D9EAF7", fontsize=6)
draw_grid(ax, pooled_context, 6.1, 3.8, .55,
          title="5-value average", color="#E2F0D9", fontsize=6)
draw_grid(ax, manual_logits, 10.0, 3.8, .32,
          title="17 logits", color="#FFF2CC", fontsize=5)
draw_grid(ax, probabilities, 16.1, 3.8, .24,
          title="17 probabilities", color="#E2F0D9", decimals=3, fontsize=4)

arrow(ax,(1.55,4.05),(2.0,4.05),"lookup")
arrow(ax,(5.25,4.05),(5.9,4.05),"mean")
arrow(ax,(9.1,4.05),(9.8,4.05),"linear")
arrow(ax,(15.65,4.05),(15.95,4.05),"softmax")

ax.text(10.5,1.1,"Every colored square contains a real scalar from this trained model.",
        ha="center",fontsize=11,color="#17365D")
plt.show()


## 13. Key interpretation

- `nn.Embedding` is a trainable lookup table.
- Two selected five-value rows are averaged into one five-value context vector.
- The linear layer contains seventeen separate output neurons, one for each vocabulary word.
- Every output neuron has five weights and one bias.
- A logit is the weighted sum for one candidate word.
- Softmax converts all logits into comparable probabilities.
- Cross-entropy rewards a high probability for the correct missing word.
- `loss.backward()` calculates gradients for both the embedding layer and output layer.
- `optimizer.step()` updates both sets of trainable parameters.
